# Day 2 — Patient Symptom Triage Classifier

**Pattern:** Prompt engineering ladder — zero-shot, few-shot, CoT, ReAct  
**Domain:** Healthcare  
**Course reference:** Lab 2

---

## What I am building today

Hospital intake desks face a constant stream of patient descriptions: 'I have chest pain and I'm sweating a lot.' Each one must be triaged into a severity bucket — Emergency, Urgent, Routine, or Self-care — within seconds. Today we build the same classifier four times using progressively richer prompting techniques, and measure how accuracy improves without changing the model.

## Learning objectives

By the end of today, you will have:

- Write the same task four ways: zero-shot, few-shot, chain-of-thought (CoT), and ReAct.
- Build a small evaluation set so you can see accuracy numerically, not feel it.
- Observe which prompting style helps most for which kinds of inputs.
- Understand when prompt engineering is enough and when you need tools (foreshadowing Day 5).


## Prerequisites

- Day 1 complete — you have a working LLM client.
- 20 labeled test cases (we'll create them in the notebook).


## Concepts to internalize before you start

- Zero-shot: just an instruction. Cheap, often surprisingly good.
- Few-shot: instruction + examples. Best price/performance for most classification.
- Chain-of-thought: ask the model to 'think step by step' before answering.
- ReAct: Reason, then Act, then observe — the foundation of tool-using agents.
- Why you must evaluate, not eyeball.


> **How to use this notebook.** Every code cell below contains only comments and TODOs — no working code. Read the markdown explanation, then write the actual implementation in the code cell, using the TODO comments as your scaffold. The point is to *write* the code, not to *run* mine. When you get stuck, the comments are your contract with yourself.

## Step 1 — Build the evaluation set

Before you can compare prompts, you need labeled data. Create 20 patient-symptom strings with the correct triage label. Mix easy cases (clear emergency, clear routine) and hard ones (ambiguous, multiple symptoms, vague language).

In [ ]:
# --- Setup: reuse the Day 1 LLM client pattern -------------------------------
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

# Search upward from the kernel's working directory so the project-root .env is
# found whether the kernel starts in Day02/ or the repo root.
_env_path = find_dotenv(usecwd=True)
load_dotenv(_env_path)
if not os.environ.get('OPENAI_API_KEY'):
    raise RuntimeError(
        f"OPENAI_API_KEY not loaded. find_dotenv resolved to: {_env_path or '(nothing)'}"
    )
openai_client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

MODEL = "gpt-4.1-mini"

# Track token usage per approach so we can compare cost later (Step 6).
_token_log = {}

def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.0, tag: str = None) -> str:
    """One chat-completion call. Returns the assistant text and logs token usage by `tag`."""
    resp = openai_client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    if tag is not None and resp.usage is not None:
        rec = _token_log.setdefault(tag, {"calls": 0, "total_tokens": 0})
        rec["calls"] += 1
        rec["total_tokens"] += resp.usage.total_tokens
    return resp.choices[0].message.content.strip()


# --- The labeled evaluation set ----------------------------------------------
# 20 cases that honor the spec: a mix of clear "floor" cases AND deliberately
# hard ones (vague, multi-symptom, conflicting-signal, second-hand). The hard
# cases are what let the four techniques actually diverge.
# NOTE: the `expected` labels on the hard cases are judgment calls — reasonable
# clinicians could disagree. That disagreement is itself part of the lesson.
dataset = [
    # --- clear cases (the floor) ---
    {'symptoms': 'crushing chest pain radiating to left arm, short of breath', 'expected': 'Emergency'},
    {'symptoms': 'sudden severe abdominal pain, vomiting, fever', 'expected': 'Emergency'},
    {'symptoms': 'dizziness, fainting, rapid heartbeat', 'expected': 'Emergency'},
    {'symptoms': 'severe burn on hand, blistering', 'expected': 'Emergency'},
    {'symptoms': 'chest tightness, difficulty breathing, sweating', 'expected': 'Emergency'},
    {'symptoms': 'sharp chest pain, difficulty speaking, weakness on one side', 'expected': 'Emergency'},
    {'symptoms': 'sprained ankle, mild swelling', 'expected': 'Routine'},
    {'symptoms': 'cut on finger from kitchen knife, bleeding stopped', 'expected': 'Routine'},
    {'symptoms': 'minor cut on forearm, slight bleeding', 'expected': 'Routine'},
    {'symptoms': 'mild headache that started this morning', 'expected': 'Self-care'},
    {'symptoms': 'runny nose, sneezing, itchy eyes', 'expected': 'Self-care'},
    {'symptoms': 'itchy rash, mild swelling', 'expected': 'Self-care'},
    {'symptoms': 'minor bruise on knee', 'expected': 'Self-care'},
    {'symptoms': 'small scrape on elbow', 'expected': 'Self-care'},
    # --- deliberately hard: vague, multi-symptom, conflicting signal ---
    {'symptoms': "just feel off, a bit dizzy and tired lately, not sure why", 'expected': 'Urgent'},
    {'symptoms': "headache for a week, now blurry vision and some nausea", 'expected': 'Urgent'},
    {'symptoms': "tummy's been weird for days, can't really describe it", 'expected': 'Routine'},
    {'symptoms': "chest pain but only when I cough, otherwise totally fine", 'expected': 'Routine'},
    {'symptoms': "on-and-off numbness in fingers for a few weeks", 'expected': 'Urgent'},
    {'symptoms': "kid has a fever, kind of fussy, hard to say how bad", 'expected': 'Urgent'},
]

# The four canonical labels. Every classifier must resolve to exactly one of these.
LABELS = ["Emergency", "Urgent", "Routine", "Self-care"]

def normalize_label(text: str) -> str:
    """Map free-form model output to one canonical label.

    Order matters: 'Self-care' is checked so a stray 'care' doesn't shadow it,
    and we match case-insensitively against the raw text.
    """
    t = text.strip().lower()
    for label in LABELS:
        if label.lower() in t:
            return label
    return text.strip()  # unrecognized — will count as a miss during evaluation


# --- The evaluation harness --------------------------------------------------
def evaluate(classifier_fn, dataset, label: str = None):
    """Run a classifier over the dataset and return (accuracy, misclassifications).

    Misclassifications are the gold for improvement — each records what the model
    predicted vs. what was expected.
    """
    correct = 0
    misclassifications = []
    for case in dataset:
        predicted = classifier_fn(case['symptoms'])
        if predicted == case['expected']:
            correct += 1
        else:
            misclassifications.append({
                'symptoms': case['symptoms'],
                'expected': case['expected'],
                'predicted': predicted,
            })
    accuracy = correct / len(dataset)
    if label:
        print(f"{label:<14} accuracy: {accuracy:.0%}  ({correct}/{len(dataset)})")
    return accuracy, misclassifications

## Step 2 — Level 1 — Zero-shot

Just instruct. No examples. Return the label. This is your floor; everything else should beat it.

In [ ]:
# Level 1 — Zero-shot: just an instruction, no examples.
def classify_zero_shot(symptoms: str) -> str:
    system = (
        "You are a triage nurse. Classify symptoms into exactly one of: "
        "Emergency, Urgent, Routine, Self-care. Reply with only the label."
    )
    out = call_llm(system, symptoms, tag="zero_shot")
    return normalize_label(out)

acc_zero, miss_zero = evaluate(classify_zero_shot, dataset, label="Zero-shot")

## Step 3 — Level 2 — Few-shot

Add 3-5 example pairs in the system prompt. The model sees the pattern and calibrates. This is usually the best cost/quality trade-off.

In [ ]:
# Level 2 — Few-shot: instruction + a few worked examples spanning all four classes.
# IMPORTANT: these examples are NOT drawn from `dataset` — using test cases as
# examples is data leakage and would inflate accuracy (see Pitfalls).
def classify_few_shot(symptoms: str) -> str:
    system = """You are a triage nurse. Classify symptoms into exactly one of:
Emergency, Urgent, Routine, Self-care. Reply with only the label.

Examples:
Symptoms: severe difficulty breathing, lips turning blue
Label: Emergency

Symptoms: deep cut on leg still bleeding after 15 minutes
Label: Urgent

Symptoms: stiff neck and mild soreness after sleeping awkwardly
Label: Routine

Symptoms: mild seasonal allergy congestion
Label: Self-care"""
    out = call_llm(system, f"Symptoms: {symptoms}\nLabel:", tag="few_shot")
    return normalize_label(out)

acc_few, miss_few = evaluate(classify_few_shot, dataset, label="Few-shot")

## Step 4 — Level 3 — Chain-of-thought (CoT)

Tell the model to reason aloud before answering. The reasoning becomes part of the output. For ambiguous medical cases this often unlocks the correct answer because the model considers severity factors it would otherwise skip.

In [ ]:
def classify_cot(symptoms: str) -> str:
    system = (
        "You are a triage nurse. Classify symptoms into exactly one of: "
        "Emergency, Urgent, Routine, Self-care.\n"
        "Think step by step about red flags and severity factors, then return the "
        "label on a final line prefixed with FINAL_ANSWER:"
    )
    out = call_llm(system, symptoms, tag="cot")

    # Extract only the text after FINAL_ANSWER: so the reasoning doesn't
    # pollute label matching. If the marker is missing, fall back to the LAST
    # label mentioned — reasoning typically rules out options before landing on
    # the answer, so the last mention is the conclusion, not the first.
    if "FINAL_ANSWER:" in out:
        verdict = out.split("FINAL_ANSWER:")[-1]
    else:
        verdict = out  # will be scanned last-match below

    # Override normalize_label's first-match behaviour: scan all matches and
    # return the last one found (the conclusion of the reasoning chain).
    t = verdict.strip().lower()
    last_match = None
    for label in LABELS:
        if label.lower() in t:
            last_match = label
    return last_match if last_match else normalize_label(verdict)

acc_cot, miss_cot = evaluate(classify_cot, dataset, label="CoT")

## Step 5 — Level 4 — ReAct (preview)

ReAct interleaves Thought, Action, Observation steps. Without tools, you can still simulate it by asking the model to 'reason about red flags, then state which red flags are present, then reach a conclusion'. We won't add real tools yet — that is Day 5 — but this gets you used to the structure.

In [ ]:
# Level 4 — ReAct-style: structured Thought -> Observation -> Answer (no real tools yet).
def classify_react_style(symptoms: str) -> str:
    system = (
        "You are a triage nurse. Reason about the case in three labeled sections, "
        "then classify into exactly one of: Emergency, Urgent, Routine, Self-care.\n"
        "Respond in EXACTLY this format:\n"
        "THOUGHT: <which red flags / severity indicators are relevant in general>\n"
        "OBSERVATION: <which of those are actually present in this case>\n"
        "ANSWER: <one label only>"
    )
    out = call_llm(system, symptoms, tag="react")
    # Pull the final label out of the ANSWER section.
    if "ANSWER:" in out:
        out = out.split("ANSWER:")[-1]
    return normalize_label(out)

acc_react, miss_react = evaluate(classify_react_style, dataset, label="ReAct-style")

## Step 6 — Compare the four approaches

Make a small table: approach, accuracy, average tokens per call, misclassifications. This is the most important output of today's lab — it teaches you to think in trade-offs, not in 'which is best'.

In [ ]:
# Step 6 — Compare the four approaches: accuracy vs. cost vs. failure modes.
def avg_tokens(tag: str) -> float:
    rec = _token_log.get(tag)
    if not rec or rec["calls"] == 0:
        return 0.0
    return rec["total_tokens"] / rec["calls"]

results = [
    ("Zero-shot",   acc_zero,  avg_tokens("zero_shot"), miss_zero),
    ("Few-shot",    acc_few,   avg_tokens("few_shot"),  miss_few),
    ("CoT",         acc_cot,   avg_tokens("cot"),       miss_cot),
    ("ReAct-style", acc_react, avg_tokens("react"),     miss_react),
]

# Comparison table
print(f"{'Approach':<14}{'Accuracy':<11}{'Avg tokens/call':<18}{'Misclassified'}")
print("-" * 58)
for name, acc, tok, miss in results:
    print(f"{name:<14}{acc:<11.0%}{tok:<18.1f}{len(miss)}")

# Per-approach misclassifications — the symptoms and what the model said.
print("\n--- Misclassifications by approach ---")
for name, acc, tok, miss in results:
    if not miss:
        print(f"\n{name}: none 🎉")
        continue
    print(f"\n{name}:")
    for m in miss:
        print(f"  • {m['symptoms'][:55]!r}")
        print(f"      expected={m['expected']}  predicted={m['predicted']}")

# Which would I ship?
# ------------------
# For a triage desk, the cost of under-triaging an Emergency is catastrophic while
# the extra tokens of CoT/ReAct are cheap by comparison — so raw accuracy on the
# ambiguous/severe cases matters more than per-call price. Few-shot is the value
# pick for clean, high-volume inputs, but I would ship the CoT (or ReAct-style)
# variant for anything touching Emergency/Urgent, because the explicit red-flag
# reasoning is what rescues the borderline cases and the reasoning trace is also
# auditable — a real plus in a clinical setting.
print("\nSee the closing comment for the ship recommendation.")

In [ ]:
# =============================================================================
# EXTENDED EVAL — 50 cases
# Run this cell AFTER the Step 6 cell so original 20-case results are preserved
# in acc_zero / acc_few / acc_cot / acc_react for the side-by-side comparison.
# =============================================================================

extra_cases = [
    # --- Emergency: clear + multi-symptom edge ---
    {'symptoms': 'unresponsive, cannot be woken up',                                          'expected': 'Emergency'},
    {'symptoms': 'severe allergic reaction, throat swelling, hives spreading fast',           'expected': 'Emergency'},
    {'symptoms': 'fell from a ladder, cannot feel or move legs',                              'expected': 'Emergency'},
    {'symptoms': 'worst headache of my life, came on suddenly out of nowhere',                'expected': 'Emergency'},
    {'symptoms': 'vomiting blood, passing dark tarry stool',                                  'expected': 'Emergency'},
    {'symptoms': 'high fever 104F with stiff neck and sensitivity to light',                  'expected': 'Emergency'},
    {'symptoms': 'baby stopped breathing for a few seconds, seems okay now but scared',       'expected': 'Emergency'},

    # --- Urgent: clear + vague/gradual onset ---
    {'symptoms': 'persistent fever 101F for five days, not coming down with Tylenol',         'expected': 'Urgent'},
    {'symptoms': 'ear pain with discharge and muffled hearing in one ear',                    'expected': 'Urgent'},
    {'symptoms': 'tooth pain keeping me up at night, gum is swollen',                        'expected': 'Urgent'},
    {'symptoms': 'blood in urine and burning sensation when urinating',                       'expected': 'Urgent'},
    {'symptoms': 'get short of breath climbing stairs, been happening for two weeks',         'expected': 'Urgent'},
    {'symptoms': 'not sure if serious but my heart seems to skip beats sometimes',            'expected': 'Urgent'},
    {'symptoms': 'feeling really anxious, shaky, not eating well, lost some weight recently', 'expected': 'Urgent'},
    {'symptoms': "arm's been sore and weak since yesterday, didn't injure it",                'expected': 'Urgent'},
    {'symptoms': 'kid has been complaining of stomach pain on and off for three days',        'expected': 'Urgent'},

    # --- Routine ---
    {'symptoms': 'ingrown toenail, a bit red and sore',                                       'expected': 'Routine'},
    {'symptoms': 'wart on finger, been there a few months',                                   'expected': 'Routine'},
    {'symptoms': 'need a routine prescription refill, feeling fine otherwise',                'expected': 'Routine'},
    {'symptoms': 'twisted knee playing basketball, sore but can walk on it',                  'expected': 'Routine'},
    {'symptoms': 'poison ivy rash, itchy but contained to one forearm',                       'expected': 'Routine'},
    {'symptoms': 'splinter deep in foot, cannot get it out myself',                           'expected': 'Routine'},
    {'symptoms': 'mild lower back pain after a long flight, no injury',                       'expected': 'Routine'},

    # --- Self-care ---
    {'symptoms': 'common cold, runny nose, mild cough, no fever',                             'expected': 'Self-care'},
    {'symptoms': 'muscle soreness all over after first gym session in months',                 'expected': 'Self-care'},
    {'symptoms': 'mild sunburn on shoulders, pink skin, no blisters',                         'expected': 'Self-care'},
    {'symptoms': 'hiccups that have lasted a couple of hours',                                 'expected': 'Self-care'},
    {'symptoms': 'mild indigestion and bloating after a heavy meal',                           'expected': 'Self-care'},
    {'symptoms': 'tension headache at the end of a very stressful day',                       'expected': 'Self-care'},
    {'symptoms': 'chapped and cracked lips, nothing else',                                     'expected': 'Self-care'},
]

dataset_50 = dataset + extra_cases
print(f"Extended dataset: {len(dataset_50)} cases "
      f"({sum(1 for c in dataset_50 if c['expected']=='Emergency')} Emergency / "
      f"{sum(1 for c in dataset_50 if c['expected']=='Urgent')} Urgent / "
      f"{sum(1 for c in dataset_50 if c['expected']=='Routine')} Routine / "
      f"{sum(1 for c in dataset_50 if c['expected']=='Self-care')} Self-care)")

# Reset token log so 50-case counts are clean
_token_log.clear()

print("\nRunning all four classifiers on 50 cases (200 API calls)...\n")
acc_zero_50,  miss_zero_50  = evaluate(classify_zero_shot,   dataset_50, label="Zero-shot")
acc_few_50,   miss_few_50   = evaluate(classify_few_shot,    dataset_50, label="Few-shot")
acc_cot_50,   miss_cot_50   = evaluate(classify_cot,         dataset_50, label="CoT")
acc_react_50, miss_react_50 = evaluate(classify_react_style, dataset_50, label="ReAct-style")

# --- Side-by-side comparison: 20-case vs 50-case ---
print(f"\n{'='*65}")
print(f"{'Approach':<14} {'20-case acc':>11}  {'50-case acc':>11}  {'Delta':>7}  {'Avg tok/call':>13}")
print(f"{'-'*65}")

def avg_tokens_50(tag):
    rec = _token_log.get(tag)
    return rec['total_tokens'] / rec['calls'] if rec and rec['calls'] else 0.0

rows = [
    ("Zero-shot",   acc_zero,  acc_zero_50,  avg_tokens_50("zero_shot")),
    ("Few-shot",    acc_few,   acc_few_50,   avg_tokens_50("few_shot")),
    ("CoT",         acc_cot,   acc_cot_50,   avg_tokens_50("cot")),
    ("ReAct-style", acc_react, acc_react_50, avg_tokens_50("react")),
]
for name, a20, a50, tok in rows:
    delta = a50 - a20
    arrow = "▲" if delta > 0 else ("▼" if delta < 0 else "─")
    print(f"{name:<14} {a20:>10.0%}   {a50:>10.0%}   {arrow}{abs(delta):>5.0%}   {tok:>12.1f}")

print(f"{'='*65}")
print("\n--- 50-case misclassifications (new cases only, to spot patterns) ---")
for name, miss in [("Zero-shot", miss_zero_50), ("Few-shot", miss_few_50),
                   ("CoT", miss_cot_50), ("ReAct-style", miss_react_50)]:
    new_misses = [m for m in miss if m in
                  [{'symptoms': c['symptoms'], 'expected': c['expected'],
                    'predicted': m['predicted']} for c in extra_cases
                   if c['symptoms'] == m['symptoms']]]
    hard_misses = [m for m in miss if m['symptoms'] in {c['symptoms'] for c in extra_cases}]
    print(f"\n{name} — {len(miss)} total misses, {len(hard_misses)} from new cases:")
    for m in hard_misses:
        print(f"  • {m['symptoms'][:58]!r}")
        print(f"      expected={m['expected']}  predicted={m['predicted']}")

In [ ]:
# =============================================================================
# STRETCH — Confidence-based cascade classifier
#
# Insight from the 50-case results: CoT costs 4x tokens but scores lower
# because it overthinks easy cases. The fix: use zero-shot as a fast first
# pass, and only escalate to CoT when confidence is below a threshold.
#
# Architecture:
#   symptoms → [zero-shot + confidence] → high conf → done
#                                       → low conf  → CoT → done
# =============================================================================
import json

CONFIDENCE_THRESHOLD = 75  # escalate to CoT if zero-shot is below this

def classify_zero_shot_confident(symptoms: str):
    """Zero-shot call that also returns a self-reported confidence 0-100.

    Asks for JSON so label and confidence come back in one call.
    Uses tag 'cascade_zs' to keep token accounting separate.
    """
    system = """You are a triage nurse. Classify symptoms into exactly one of:
Emergency, Urgent, Routine, Self-care.

Reply with JSON only — no markdown, no explanation:
{"label": "<one of the four labels>", "confidence": <integer 0-100>}"""
    out = call_llm(system, symptoms, tag="cascade_zs")
    try:
        data = json.loads(out)
        label = normalize_label(str(data.get("label", "")))
        confidence = int(data.get("confidence", 0))
        return label, confidence
    except Exception:
        # JSON parse failed — treat as low confidence so CoT takes over
        return normalize_label(out), 0


def classify_cascade(symptoms: str, threshold: int = CONFIDENCE_THRESHOLD):
    """Route to CoT only when zero-shot confidence is below threshold.

    Returns (label, tier, confidence) so we can audit which cases escalated.
    Uses tag 'cascade_cot' — separate from the flat CoT run — so token
    counts stay clean.
    """
    label, confidence = classify_zero_shot_confident(symptoms)
    if confidence >= threshold:
        return label, "zero-shot", confidence

    # Escalate: same CoT logic as classify_cot but with its own tag
    system = (
        "You are a triage nurse. Classify symptoms into exactly one of: "
        "Emergency, Urgent, Routine, Self-care.\n"
        "Think step by step about red flags and severity factors, then return "
        "the label on a final line prefixed with FINAL_ANSWER:"
    )
    out = call_llm(system, symptoms, tag="cascade_cot")
    if "FINAL_ANSWER:" in out:
        verdict = out.split("FINAL_ANSWER:")[-1]
    else:
        verdict = out
    t = verdict.strip().lower()
    last_match = None
    for lbl in LABELS:
        if lbl.lower() in t:
            last_match = lbl
    return (last_match if last_match else normalize_label(verdict)), "cot", confidence


# --- Run cascade on dataset_50 -----------------------------------------------
print(f"Running cascade on {len(dataset_50)} cases "
      f"(threshold = {CONFIDENCE_THRESHOLD}% confidence)...\n")

cascade_results = []
for case in dataset_50:
    label, tier, conf = classify_cascade(case['symptoms'])
    cascade_results.append({
        'symptoms':  case['symptoms'],
        'expected':  case['expected'],
        'predicted': label,
        'tier':      tier,
        'confidence': conf,
    })

# --- Accuracy ----------------------------------------------------------------
correct = sum(1 for r in cascade_results if r['predicted'] == r['expected'])
acc_cascade = correct / len(cascade_results)

zs_results  = [r for r in cascade_results if r['tier'] == 'zero-shot']
cot_results = [r for r in cascade_results if r['tier'] == 'cot']

zs_acc  = sum(1 for r in zs_results  if r['predicted'] == r['expected']) / len(zs_results)  if zs_results  else 0
cot_acc = sum(1 for r in cot_results if r['predicted'] == r['expected']) / len(cot_results) if cot_results else 0

# --- Token cost (blended) ----------------------------------------------------
zs_rec  = _token_log.get("cascade_zs",  {"calls": 0, "total_tokens": 0})
cot_rec = _token_log.get("cascade_cot", {"calls": 0, "total_tokens": 0})

total_tokens = zs_rec["total_tokens"] + cot_rec["total_tokens"]
blended_avg  = total_tokens / len(dataset_50)
flat_zs_avg  = _token_log.get("zero_shot", {}).get("total_tokens", 0) / max(_token_log.get("zero_shot", {}).get("calls", 1), 1)
flat_cot_avg = _token_log.get("cot",       {}).get("total_tokens", 0) / max(_token_log.get("cot",       {}).get("calls", 1), 1)

# --- Summary table -----------------------------------------------------------
print(f"{'='*62}")
print(f"{'Approach':<20} {'50-case acc':>11}  {'Avg tok/call':>13}")
print(f"{'-'*62}")
print(f"{'Zero-shot (flat)':<20} {acc_zero_50:>10.0%}   {flat_zs_avg:>12.1f}")
print(f"{'CoT (flat)':<20} {acc_cot_50:>10.0%}   {flat_cot_avg:>12.1f}")
print(f"{'Cascade ←':<20} {acc_cascade:>10.0%}   {blended_avg:>12.1f}  ← blended")
print(f"{'='*62}")

print(f"\nCascade routing breakdown:")
print(f"  Zero-shot tier : {len(zs_results):>3} cases ({len(zs_results)/len(dataset_50):.0%})  accuracy = {zs_acc:.0%}")
print(f"  CoT tier       : {len(cot_results):>3} cases ({len(cot_results)/len(dataset_50):.0%})  accuracy = {cot_acc:.0%}")
print(f"  Token savings vs flat CoT: "
      f"{(flat_cot_avg - blended_avg) / flat_cot_avg:.0%} fewer tokens per call")

# --- Which cases escalated? --------------------------------------------------
print(f"\n--- Cases escalated to CoT (confidence < {CONFIDENCE_THRESHOLD}%) ---")
for r in cot_results:
    hit = "✅" if r['predicted'] == r['expected'] else "❌"
    print(f"  {hit} conf={r['confidence']:>3}%  {r['symptoms'][:52]!r}")
    if r['predicted'] != r['expected']:
        print(f"       expected={r['expected']}  predicted={r['predicted']}")

# --- Misclassifications in the cascade ---------------------------------------
misses = [r for r in cascade_results if r['predicted'] != r['expected']]
print(f"\n--- Cascade misses ({len(misses)} total) ---")
for r in misses:
    print(f"  [{r['tier']:<9}] {r['symptoms'][:52]!r}")
    print(f"             expected={r['expected']}  predicted={r['predicted']}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── data ──────────────────────────────────────────────────────────────────────
labels   = ["Zero-shot", "Few-shot", "CoT", "ReAct-style"]
acc_20   = [acc_zero*100,    acc_few*100,    acc_cot*100,    acc_react*100]
acc_50   = [acc_zero_50*100, acc_few_50*100, acc_cot_50*100, acc_react_50*100]
deltas   = [a50 - a20 for a20, a50 in zip(acc_20, acc_50)]

def _tok(tag):
    rec = _token_log.get(tag, {"calls": 1, "total_tokens": 0})
    return rec["total_tokens"] / max(rec["calls"], 1)
tokens = [_tok("zero_shot"), _tok("few_shot"), _tok("cot"), _tok("react")]

# ── palette ───────────────────────────────────────────────────────────────────
BG       = "#0D1117"
PANEL    = "#161B22"
COLORS   = ["#58A6FF", "#3FB950", "#FF7B72", "#D2A8FF"]   # blue / green / red / purple
DARK     = ["#1C3A5E", "#163C20", "#4A1C1C", "#3A1F5E"]   # muted versions for 20-case bars
GOLD     = "#F0B429"
WHITE    = "#E6EDF3"
GREY     = "#8B949E"

x   = np.arange(len(labels))
w   = 0.35

fig, axes = plt.subplots(1, 3, figsize=(17, 6.5))
fig.patch.set_facecolor(BG)
fig.suptitle(
    "Day 2 — Prompt Engineering Ladder: Results",
    fontsize=15, fontweight="bold", color=WHITE, y=1.01
)

def style_ax(ax, title):
    ax.set_facecolor(PANEL)
    ax.set_title(title, color=WHITE, fontsize=11, fontweight="bold", pad=10)
    ax.tick_params(colors=GREY, labelsize=9)
    ax.spines[["top","right","left","bottom"]].set_visible(False)
    ax.yaxis.grid(True, color="#21262D", linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for label in ax.get_xticklabels():
        label.set_color(WHITE)
        label.set_fontsize(8.5)

# ── panel 1 — accuracy 20 vs 50 cases ────────────────────────────────────────
ax1 = axes[0]
style_ax(ax1, "Accuracy — 20-case vs 50-case")

bars_20 = ax1.bar(x - w/2, acc_20, w, color=DARK,   label="20 cases", zorder=3,
                  edgecolor=COLORS, linewidth=1.2)
bars_50 = ax1.bar(x + w/2, acc_50, w, color=COLORS, label="50 cases", zorder=3,
                  edgecolor="none")

for bar, val in zip(bars_20, acc_20):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.8, f"{val:.0f}%",
             ha="center", va="bottom", color=WHITE, fontsize=8, fontweight="bold")
for bar, val in zip(bars_50, acc_50):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.8, f"{val:.0f}%",
             ha="center", va="bottom", color=WHITE, fontsize=8, fontweight="bold")

ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=12, ha="right")
ax1.set_ylim(0, 105)
ax1.set_ylabel("Accuracy (%)", color=GREY, fontsize=9)
ax1.tick_params(axis="y", colors=GREY)

legend_patches = [
    mpatches.Patch(facecolor="#2A2D35", edgecolor=WHITE, linewidth=1, label="20 cases"),
    mpatches.Patch(facecolor=WHITE, label="50 cases"),
]
ax1.legend(handles=legend_patches, loc="lower right",
           framealpha=0, labelcolor=WHITE, fontsize=8)

# ── panel 2 — delta ──────────────────────────────────────────────────────────
ax2 = axes[1]
style_ax(ax2, "Accuracy Delta (50-case − 20-case)")

delta_colors = [GOLD if d == max(deltas) else c for d, c in zip(deltas, COLORS)]
bars_d = ax2.bar(x, deltas, 0.5, color=delta_colors, zorder=3, edgecolor="none")

for bar, val, d_col in zip(bars_d, deltas, delta_colors):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.15, f"+{val:.0f}pp",
             ha="center", va="bottom", color=d_col, fontsize=9, fontweight="bold")

ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=12, ha="right")
ax2.set_ylim(0, max(deltas) + 5)
ax2.set_ylabel("Percentage points gained", color=GREY, fontsize=9)
ax2.tick_params(axis="y", colors=GREY)

# highlight the winner annotation
winner_idx = deltas.index(max(deltas))
ax2.annotate("Most improved", xy=(winner_idx, max(deltas)),
             xytext=(winner_idx + 0.6, max(deltas) + 2),
             color=GOLD, fontsize=8, fontweight="bold",
             arrowprops=dict(arrowstyle="->", color=GOLD, lw=1.2))

# ── panel 3 — avg tokens/call ─────────────────────────────────────────────────
ax3 = axes[2]
style_ax(ax3, "Avg Tokens / Call (50-case run)")

bars_t = ax3.bar(x, tokens, 0.5, color=COLORS, zorder=3, edgecolor="none")

for bar, val in zip(bars_t, tokens):
    ax3.text(bar.get_x() + bar.get_width()/2, val + 2, f"{val:.0f}",
             ha="center", va="bottom", color=WHITE, fontsize=9, fontweight="bold")

# cost ratio annotation on CoT bar
cot_idx = labels.index("CoT")
ratio = tokens[cot_idx] / tokens[0]
ax3.annotate(f"~{ratio:.0f}× zero-shot", xy=(cot_idx, tokens[cot_idx]),
             xytext=(cot_idx + 0.55, tokens[cot_idx] - 30),
             color="#FF7B72", fontsize=8, fontweight="bold",
             arrowprops=dict(arrowstyle="->", color="#FF7B72", lw=1.2))

ax3.set_xticks(x)
ax3.set_xticklabels(labels, rotation=12, ha="right")
ax3.set_ylim(0, max(tokens) * 1.22)
ax3.set_ylabel("Total tokens (prompt + completion)", color=GREY, fontsize=9)
ax3.tick_params(axis="y", colors=GREY)

# ── footer ───────────────────────────────────────────────────────────────────
fig.text(0.5, -0.04,
         "gpt-4.1-mini · temperature=0 · 50 labeled triage cases (Emergency / Urgent / Routine / Self-care)",
         ha="center", color=GREY, fontsize=8)

plt.tight_layout(pad=2.5)
plt.savefig("day02_results.png", dpi=160, bbox_inches="tight",
            facecolor=BG, edgecolor="none")
plt.show()
print("Saved → day02_results.png")

## Self-check — answer these in writing before moving on

- Did CoT always improve accuracy, or did it sometimes hurt? Why might it hurt on easy cases?
- Which is more expensive: a single CoT call, or three zero-shot calls voted? Which is more accurate?
- How would you measure 'safe failure' — i.e., the model saying 'I'm not sure, escalate' instead of guessing?


## Pitfalls to avoid

- Tiny eval sets (5 examples) give you noise, not signal. Twenty is a bare minimum.
- Mixing test cases into your few-shot examples — that is data leakage; your accuracy will be inflated.
- Comparing only accuracy without comparing token cost. CoT is 3-5x more expensive per call.


## Your LinkedIn post for today

Copy this as your starting point. Personalize it with your actual numbers, your actual frustrations, and your actual surprises from the build. The hook line is what people see in their feed — make it count.

---

**Day 2 of 14: Same model. Four prompts. Watched the accuracy climb 40 points.**

Built a patient symptom triage classifier four times today, using the same LLM but different prompting techniques: zero-shot, few-shot, chain-of-thought, and a ReAct-style structured-reasoning prompt.

I used a 20-case eval set (Emergency / Urgent / Routine / Self-care) with deliberately ambiguous cases mixed in. Then I measured each approach the same way.

What I learned:
- Zero-shot is the floor. Everything else should beat it or you're doing something wrong.
- Few-shot was the best cost/quality trade-off for clean cases.
- Chain-of-thought rescued the ambiguous ones — but costs ~3x more tokens.
- The ReAct structure (Thought → Observation → Answer) was the closest thing to how a real triage nurse actually thinks.

Bigger takeaway: the difference between an okay GenAI engineer and a good one is measuring. Anyone can write a prompt. Few people benchmark four versions of the same prompt against a held-out set.

Tomorrow: RAG. Time to give the model some documents to work with.

#PromptEngineering #LLM #Healthcare #AgenticAI #LearningInPublic

---

**Posting checklist:**

- [ ] Add a screenshot of your terminal output, the graph diagram, or a code snippet
- [ ] Replace generic numbers with your own (token counts, latency, accuracy)
- [ ] If you have a GitHub repo, link it in the FIRST comment (LinkedIn deprioritizes posts with external links)
- [ ] Tag 1-2 people who might find it useful — gentle, not spammy
- [ ] Post between 8am and 10am your timezone for best reach

## Stretch goals (only if you finish with time to spare)

- Add error handling around every external call.
- Write 5 more test cases that you didn't think of in the main flow.
- Refactor: can you make this work with a *different* LLM provider with minimal changes? That tells you whether your abstraction is right.
- Document your design decisions in a short README — your future interviewer will read it.